<a href="https://colab.research.google.com/github/kr1stawang0510-cloud/individual-assignment/blob/main/What_Weather_Conditions_Drive_Extreme_Fire_Risk_in_Australia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What Weather Conditions Drive Extreme Fire Risk in Australia?

**Individual Project — EMSC2010**

---

## 1. Problem Framing

Australia experiences some of the most severe bushfires in the world. The 2019–2020 'Black Summer' season burned over 18 million hectares and caused widespread ecological and human damage. Understanding *which meteorological conditions* create extreme fire risk is critical for early warning systems and disaster preparedness.

This project investigates the following research question:

> **Can temperature, rainfall, and humidity predict the occurrence of extreme fire-risk weather days in Australia, and which factor is most influential?**

I define an **extreme fire-risk day** as one where maximum temperature exceeds the 90th percentile AND rainfall is below the 10th percentile — a compound threshold that captures the dangerous combination of heat and dryness known to drive bushfire behaviour.

### Scope

- **Data:** Two datasets are used:
  1. `weatherAUS.csv` — Daily weather observations from Australian
     Bureau of Meteorology stations across Australia, covering 2007–2017
  2. `IDCJAC0009_066214_2019_Data.csv` and `IDCJAC0009_066214_2020_Data.csv`
     — Official daily rainfall records for Sydney Observatory Hill
     station (Station 066214), covering 2019–2020, downloaded directly
     from the Australian Bureau of Meteorology
- **Methods:** EDA, Bayesian inference (PyMC), Bayesian regression
  with model comparison (Bambi + LOO-CV), and Bootstrapping.
- **Limitation:** Fire-risk is approximated from weather conditions
  alone; actual ignition, fuel load, and wind are not fully captured.

## 2. Setup

Here we import all the libraries needed for the analysis.
- numpy and pandas: for data manipulation
- matplotlib: for visualisation
- pymc and arviz: for Bayesian inference
- bambi: for Bayesian regression

In [1]:
# Import all required libraries
!pip install pymc arviz bambi -q

import numpy as np                       # numerical operations
import pandas as pd                       # data manipulation
import matplotlib.pyplot as plt                 # plotting
import pymc as pm                        # Bayesian inference
import arviz as az                       # posterior analysis and LOO-CV
import bambi as bmb                       # Bayesian regression
import warnings
warnings.filterwarnings('ignore')                 # suppress minor warnings

# Set random seed so results are reproducible
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.6/109.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.9/218.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 6.6 MB/s eta 0:00:00


## 3. Data Cleaning and Understanding

Before any analysis, I inspect the data for missing values, data types, and potential issues.


In [2]:
df = pd.read_csv('weatherAUS.csv')

# Preview the data
print(f"Dataset shape: {df.shape}")          # show number of rows and columns
print(f"Columns: {list(df.columns)}")         # show all column names
df.head()                       # display first 5 rows

FileNotFoundError: [Errno 2] No such file or directory: 'weatherAUS.csv'

### 3.1 Checking Missing Values

Before cleaning, we inspect which columns have missing data
and how much. This informs which columns we can safely use
and whether dropping rows is reasonable.

In [ ]:
# Calculate missing value counts and percentages for each column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

# Combine into a summary table
missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

# Show only columns that have at least one missing value
print(missing_summary[missing_summary['Missing Count'] > 0]
      .sort_values('Missing %', ascending=False))

In [ ]:
key_cols = ['Date', 'Location', 'MaxTemp', 'MinTemp', 'Rainfall',
            'Humidity9am', 'Humidity3pm', 'WindSpeed9am']
df_clean = df[key_cols].dropna().copy()

df_clean['Date'] = pd.to_datetime(df_clean['Date'])
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month

print(f"Clean dataset: {df_clean.shape[0]} rows")
print(f"Date range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
print(f"Number of weather stations: {df_clean['Location'].nunique()}")
df_clean.describe().round(2)

### 3.2 Cleaning the Data

We select only the columns relevant to our analysis:
- MaxTemp: maximum daily temperature
- MinTemp: minimum daily temperature  
- Rainfall: daily precipitation in mm
- Humidity3pm: afternoon humidity (more relevant to fire risk than morning)
- WindSpeed9am: wind speed (contributes to fire spread)

We then drop any rows where these key variables are missing.

In [ ]:
# Select only the columns we need for analysis
key_cols = ['Date', 'Location', 'MaxTemp', 'MinTemp',
            'Rainfall', 'Humidity9am', 'Humidity3pm', 'WindSpeed9am']
df_clean = df[key_cols].dropna().copy()   # drop rows with any missing values

# Parse the Date column into datetime format
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# Extract year and month for later grouping
df_clean['Year'] = df_clean['Date'].dt.year
df_clean['Month'] = df_clean['Date'].dt.month

# Summary statistics
print(f"Clean dataset: {df_clean.shape[0]} rows")
print(f"Date range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
print(f"Number of weather stations: {df_clean['Location'].nunique()}")
df_clean.describe().round(2)              # show summary statistics for all columns


### 3.3 Defining the Extreme Fire-Risk Label

Since we do not have direct fire occurrence data, we construct
a weather-based proxy label called `ExtremeFire`.

A day is labelled as extreme fire-risk (1) if:
- MaxTemp > 90th percentile of all MaxTemp values (very hot day), AND
- Rainfall = 0mm (no rain at all)

**Why these thresholds?**
- The 90th percentile captures genuinely extreme heat, not just warm days
- Zero rainfall is used instead of the 10th percentile because the
  10th percentile of Australian rainfall is 0mm (many days have no rain),
  making `Rainfall < 0` always false

**Acknowledged limitation:** This label approximates fire danger from
weather alone. Actual fire occurrence also depends on fuel load,
ignition sources, and terrain — none of which are in this dataset.

In [ ]:
# Calculate the 90th percentile threshold for maximum temperature
temp_threshold = df_clean['MaxTemp'].quantile(0.90)

print(f"MaxTemp 90th percentile: {temp_threshold:.1f}°C")
print(f"Days with zero rainfall: {(df_clean['Rainfall'] == 0).sum()}")

# Define extreme fire-risk days:
# Hot (above 90th percentile) AND completely dry (0mm rainfall)
df_clean['ExtremeFire'] = (
    (df_clean['MaxTemp'] > temp_threshold) &   # very hot day
    (df_clean['Rainfall'] == 0)                # no rainfall at all
).astype(int)                                  # convert True/False to 1/0

# Check the distribution of the label
n_extreme = df_clean['ExtremeFire'].sum()
pct_extreme = n_extreme / len(df_clean) * 100
print(f"\nExtreme fire-risk days: {n_extreme} ({pct_extreme:.1f}% of all days)")
print(df_clean['ExtremeFire'].value_counts())

## 4. Exploratory Data Analysis

Before modelling, we visualise the data to understand:
1. How weather variables differ between extreme fire-risk and normal days
2. Whether fire risk follows a seasonal pattern (expected to peak in summer)
3. Whether the annual proportion of fire-risk days is changing over time

These plots motivate the modelling choices in Section 5 and 6.

In [ ]:
# Figure 1a: Distribution of MaxTemp for fire-risk vs normal days
plt.figure(figsize=(7, 4))

for risk_val, name, color in [(1, 'Extreme Risk', 'firebrick'),
                               (0, 'Normal', 'steelblue')]:
    subset = df_clean[df_clean['ExtremeFire'] == risk_val]['MaxTemp']
    plt.hist(subset, bins=40, alpha=0.5, color=color,
             label=name, density=True)            # density=True normalises the histogram

plt.xlabel('Max Temperature (°C)')
plt.ylabel('Density')
plt.title('Max Temperature: Extreme Fire-Risk vs Normal Days')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
# Expected: extreme risk days should be concentrated at higher temperatures


In [ ]:
# Figure 1b: Distribution of Rainfall for fire-risk vs normal days
plt.figure(figsize=(7, 4))

for risk_val, name, color in [(1, 'Extreme Risk', 'firebrick'),
                               (0, 'Normal', 'steelblue')]:
    subset = df_clean[df_clean['ExtremeFire'] == risk_val]['Rainfall']
    plt.hist(subset, bins=40, alpha=0.5, color=color,
             label=name, density=True)            # density=True for fair comparison

plt.xlabel('Rainfall (mm)')
plt.ylabel('Density')
plt.title('Rainfall: Extreme Fire-Risk vs Normal Days')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
# Expected: extreme risk days all have 0mm rainfall by definition

In [ ]:
# Figure 1c: Distribution of Humidity3pm for fire-risk vs normal days
plt.figure(figsize=(7, 4))

for risk_val, name, color in [(1, 'Extreme Risk', 'firebrick'),
                               (0, 'Normal', 'steelblue')]:
    subset = df_clean[df_clean['ExtremeFire'] == risk_val]['Humidity3pm']
    plt.hist(subset, bins=40, alpha=0.5, color=color,
             label=name, density=True)            # afternoon humidity more relevant to fire

plt.xlabel('Afternoon Humidity (%)')
plt.ylabel('Density')
plt.title('Afternoon Humidity: Extreme Fire-Risk vs Normal Days')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
# Expected: extreme risk days should have lower afternoon humidity

### 4.2 Seasonal Pattern

Australia's bushfire season peaks in summer (December–February)
when temperatures are highest and rainfall is lowest.

Here we check whether our fire-risk label correctly captures
this known seasonal pattern — if it does, it validates our
threshold definition.

In [ ]:
# Figure 2: Monthly proportion of extreme fire-risk days
monthly_risk = df_clean.groupby('Month')['ExtremeFire'].mean() * 100
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(10, 4))
plt.bar(month_names, monthly_risk.values,
        color='firebrick', alpha=0.8)             # bar chart by month
plt.xlabel('Month')
plt.ylabel('Extreme Fire-Risk Days (%)')
plt.title('Seasonal Pattern of Extreme Fire-Risk Days in Australia')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print the peak month
print("Peak fire-risk month:", month_names[monthly_risk.idxmax() - 1])

### 4.3 Annual Trend

We also check whether the proportion of extreme fire-risk days
is increasing over the study period (2007–2017).

An upward trend would suggest that dangerous fire weather is
becoming more frequent — consistent with the broader pattern
of climate change increasing fire risk in Australia.

In [ ]:
# Figure 3: Annual proportion of extreme fire-risk days
annual_risk = df_clean.groupby('Year')['ExtremeFire'].mean() * 100

plt.figure(figsize=(10, 4))
plt.plot(annual_risk.index, annual_risk.values,
         'o-', color='firebrick', linewidth=2)    # line plot with markers
plt.xlabel('Year')
plt.ylabel('Extreme Fire-Risk Days (%)')
plt.title('Annual Proportion of Extreme Fire-Risk Days (2007–2017)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Bayesian Analysis: Do High-Temperature Days Have Less Rainfall?

Before building a full predictive model, we use a Bayesian
two-group comparison (as used in class Week 7) to test whether
days with extreme maximum temperatures genuinely receive less rainfall.

This confirms the physical mechanism behind our fire-risk definition
and follows the same PyMC structure used in the course notebooks.

**Model structure:**
- Two groups: high-temperature days vs normal days
- Priors: Normal for means, HalfNormal for standard deviations
- Likelihood: Normal distribution
- Quantity of interest: the difference in mean rainfall between groups
- If the 95% HDI excludes zero → strong evidence of a real difference

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

# Split into two groups based on temperature threshold
high_temp_days = df_clean[df_clean['MaxTemp'] > temp_threshold]['Rainfall'].values
normal_days    = df_clean[df_clean['MaxTemp'] <= temp_threshold]['Rainfall'].values

# Sample 500 from each group to keep PyMC sampling fast
high_temp_sample = rng.choice(high_temp_days, size=500, replace=False)
normal_sample    = rng.choice(normal_days,    size=500, replace=False)

print(f"High-temp days mean rainfall: {high_temp_sample.mean():.2f} mm")
print(f"Normal days mean rainfall:    {normal_sample.mean():.2f} mm")
# We expect high-temp days to have lower mean rainfall

In [ ]:
# Bayesian two-group model (same structure as course Week 7)
with pm.Model() as rainfall_model:

    # Priors for the group means (weakly informative)
    mu_high   = pm.Normal('mu_high',   mu=high_temp_sample.mean(), sigma=10)
    mu_normal = pm.Normal('mu_normal', mu=normal_sample.mean(),    sigma=10)

    # Priors for standard deviations (must be positive)
    sigma_high   = pm.HalfNormal('sigma_high',   sigma=10)
    sigma_normal = pm.HalfNormal('sigma_normal', sigma=10)

    # Likelihood: observed data given the model parameters
    y_high   = pm.Normal('y_high',   mu=mu_high,
                          sigma=sigma_high,   observed=high_temp_sample)
    y_normal = pm.Normal('y_normal', mu=mu_normal,
                          sigma=sigma_normal, observed=normal_sample)

    # Deterministic variable: the difference in means
    # This is what we actually want to analyse
    diff_of_means = pm.Deterministic('difference', mu_high - mu_normal)

    # Sample the posterior distribution
    trace = pm.sample(2000, return_inferencedata=True,
                      target_accept=0.95, random_seed=RANDOM_SEED)

In [ ]:
# Plot the posterior distribution of the difference in means
az.plot_posterior(trace, var_names=['difference'],
                  ref_val=0,          # orange line at zero = no difference
                  hdi_prob=0.95)      # show 95% credible interval
plt.title('Bayesian Posterior: Difference in Mean Rainfall\n'
          '(High-Temperature Days minus Normal Days)')
plt.xlabel('Difference in Mean Rainfall (mm)')
plt.ylabel('Posterior Density')
plt.show()

# Print numerical summary
summary = az.summary(trace, var_names=['mu_high', 'mu_normal', 'difference'])
print(summary)
# If HDI of 'difference' excludes zero: strong evidence high-temp days are drier

## 6. Bayesian Regression: Which Variables Best Predict Fire Risk?

Using Bambi (as used in class Week 8), we fit four logistic
regression models to predict `ExtremeFire` (1 = fire-risk day,
0 = normal day) from weather variables.

We then compare these models using LOO-CV (Leave-One-Out
Cross-Validation) via ArviZ — the same method used in the
course Week 8 notebook — to identify which combination of
predictors is most informative.

**Why logistic regression?**
Our outcome variable is binary (0 or 1), so logistic regression
is the appropriate model family.

**Why standardise predictors?**
Standardising (z-scoring) puts all predictors on the same scale,
allowing direct comparison of coefficient sizes, and improves
sampling efficiency in PyMC.

**The four models:**
1. Temperature only
2. Temperature + Rainfall
3. Temperature + Rainfall + Humidity
4. All four predictors (+ Wind Speed)

In [ ]:
# Select predictors and outcome variable
model_df = df_clean[['MaxTemp', 'Rainfall', 'Humidity3pm',
                      'WindSpeed9am', 'ExtremeFire']].dropna().copy()

# Standardise all predictors (z-score: subtract mean, divide by std)
for col in ['MaxTemp', 'Rainfall', 'Humidity3pm', 'WindSpeed9am']:
    model_df[f'{col}_scaled'] = (
        (model_df[col] - model_df[col].mean()) / model_df[col].std()
    )                                             # scaled version of each predictor

# Sample 2000 rows for computational speed
model_sample = model_df.sample(n=2000, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Modelling sample: {len(model_sample)} rows")
print(f"Extreme fire-risk days in sample: {model_sample['ExtremeFire'].sum()}")

In [ ]:
# Model 1: Temperature only (simplest model)
model_temp = bmb.Model('ExtremeFire ~ MaxTemp_scaled',
                        model_sample, family='bernoulli')  # bernoulli = logistic
idata_temp = model_temp.fit(idata_kwargs={'log_likelihood': True},
                             random_seed=RANDOM_SEED)

# Model 2: Temperature + Rainfall
model_temp_rain = bmb.Model('ExtremeFire ~ MaxTemp_scaled + Rainfall_scaled',
                             model_sample, family='bernoulli')
idata_temp_rain = model_temp_rain.fit(idata_kwargs={'log_likelihood': True},
                                       random_seed=RANDOM_SEED)

# Model 3: Temperature + Rainfall + Humidity (expected best model)
model_full = bmb.Model('ExtremeFire ~ MaxTemp_scaled + Rainfall_scaled + Humidity3pm_scaled',
                        model_sample, family='bernoulli')
idata_full = model_full.fit(idata_kwargs={'log_likelihood': True},
                             random_seed=RANDOM_SEED)

# Model 4: All four predictors
model_all = bmb.Model(
    'ExtremeFire ~ MaxTemp_scaled + Rainfall_scaled + Humidity3pm_scaled + WindSpeed9am_scaled',
    model_sample, family='bernoulli')
idata_all = model_all.fit(idata_kwargs={'log_likelihood': True},
                           random_seed=RANDOM_SEED)

In [ ]:
# Compare all four models using LOO-CV (same method as course Week 8)
comparison = az.compare({
    'temp_only':       idata_temp,        # simplest model
    'temp_rain':       idata_temp_rain,
    'temp_rain_humid': idata_full,        # expected winner
    'all_predictors':  idata_all          # most complex model
})
print(comparison)                         # higher elpd_loo = better model

# Visualise the comparison
az.plot_compare(comparison, insample_dev=False)
plt.title('LOO-CV Model Comparison: Predicting Extreme Fire-Risk Days')
plt.tight_layout()
plt.show()
# The model at the top is the best-performing one

In [ ]:
# Posterior distribution of the intercept
# Intercept represents the baseline log-odds of fire risk
az.plot_posterior(idata_full,
                  var_names=['Intercept'],
                  hdi_prob=0.95)
plt.title('Posterior: Intercept (Baseline Fire Risk)')
plt.show()

In [ ]:
# Posterior distribution of MaxTemp coefficient
# Positive value = higher temperature increases fire risk probability
az.plot_posterior(idata_full,
                  var_names=['MaxTemp_scaled'],
                  hdi_prob=0.95)
plt.title('Posterior: MaxTemp Coefficient')
plt.show()

In [ ]:
# Posterior distribution of Rainfall coefficient
# Negative value = more rainfall decreases fire risk probability
az.plot_posterior(idata_full,
                  var_names=['Rainfall_scaled'],
                  hdi_prob=0.95)
plt.title('Posterior: Rainfall Coefficient')
plt.show()

In [ ]:
# Posterior distribution of Humidity coefficient
# Negative value = higher humidity decreases fire risk probability
az.plot_posterior(idata_full,
                  var_names=['Humidity3pm_scaled'],
                  hdi_prob=0.95)
plt.title('Posterior: Humidity3pm Coefficient')
plt.show()

## 6.5 Validation: Rainfall Patterns During the 2019–2020 Black Summer

Our model was trained on data up to 2017. To assess whether the
weather conditions during the 2019–2020 Black Summer were genuinely
extreme, we download official daily rainfall data from the Australian
Bureau of Meteorology (BOM) for Sydney Observatory Hill station
(Station 066214).

This allows us to ask:
> Were the rainfall conditions during the 2019–2020 Black Summer
> significantly lower than historical averages, consistent with
> our model's definition of extreme fire-risk days?

In [ ]:
import pandas as pd

# Load BOM daily rainfall data for Sydney Observatory Hill Station 066214
# Source: Australian Bureau of Meteorology
df_2019 = pd.read_csv('IDCJAC0009_066214_2019_Data.csv')
df_2020 = pd.read_csv('IDCJAC0009_066214_2020_Data.csv')

# Combine both years into one dataframe
df_rain = pd.concat([df_2019, df_2020], ignore_index=True)

print(f"Total rows: {len(df_rain)}")
print(f"Columns: {df_rain.columns.tolist()}")
print(df_rain.head())

In [ ]:
# Create proper date column
df_rain['Date'] = pd.to_datetime(df_rain[['Year', 'Month', 'Day']])

# Rename rainfall column for easier use
df_rain = df_rain.rename(columns={
    'Rainfall amount (millimetres)': 'Rainfall'
})

# Filter to Black Summer period (Sep 2019 - Mar 2020)
black_summer = df_rain[
    (df_rain['Date'] >= '2019-09-01') &
    (df_rain['Date'] <= '2020-03-31')
].copy()

# Compare with historical average from weatherAUS.csv
sydney_historical = df_clean[df_clean['Location'] == 'Sydney']['Rainfall'].mean()
black_summer_avg = black_summer['Rainfall'].mean()

print(f"Historical daily rainfall average (2007-2017): {sydney_historical:.2f} mm")
print(f"Black Summer daily rainfall average (2019-2020): {black_summer_avg:.2f} mm")
print(f"Reduction: {((sydney_historical - black_summer_avg)/sydney_historical*100):.1f}%")

# Plot rainfall during Black Summer
plt.figure(figsize=(12, 4))
plt.bar(black_summer['Date'], black_summer['Rainfall'],
        color='steelblue', alpha=0.7,
        label='Daily Rainfall')
plt.axhline(y=sydney_historical, color='firebrick',
            linestyle='--', linewidth=2,
            label=f'Historical average ({sydney_historical:.1f} mm)')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm)')
plt.title('Daily Rainfall in Sydney During the 2019–2020 Black Summer\n'
          'vs Historical Average (2007–2017)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretation

The plot reveals a striking pattern consistent with the known
climatology of the 2019–2020 Black Summer:

- From September to December 2019, daily rainfall was almost
  entirely absent, with most days recording 0mm — well below
  the historical average of 3.34mm
- This extended dry period created the dangerous conditions
  that allowed fires to ignite and spread across NSW
- The heavy rainfall in February 2020 effectively ended the
  fire season

Although the overall Black Summer average (3.80mm) appears
slightly above the historical mean due to the February rainfall
events, the September–December 2019 period represents an
extreme and sustained rainfall deficit consistent with our
model's definition of extreme fire-risk conditions.

In [ ]:
# Numerical summary of all coefficients
# hdi_2.5% and hdi_97.5% = 95% credible interval
# If HDI excludes zero = strong evidence this variable matters
print(az.summary(idata_full,
                 var_names=['Intercept', 'MaxTemp_scaled',
                            'Rainfall_scaled', 'Humidity3pm_scaled']))

### Interpretation of Coefficients

All three predictors show strong evidence of a real effect
(95% HDI excludes zero for all variables):

- **MaxTemp (mean=5.17):** The strongest predictor — higher
  temperatures dramatically increase fire-risk probability
- **Rainfall (mean=-2.36):** More rainfall strongly reduces
  fire risk, as expected
- **Humidity3pm (mean=-0.41):** Higher afternoon humidity
  moderately reduces fire risk

Since all predictors are standardised (z-scored), the coefficient
magnitudes can be directly compared. MaxTemp has by far the largest
effect, confirming it as the dominant driver of extreme fire-risk
days in Australia.

## 7. Bootstrapping: How Reliable is the Seasonal Pattern?

The seasonal pattern in Section 4 was computed from sample data.
To assess whether this pattern is robust or could be due to
chance variation, we use bootstrapping (as used in class Week 5)
to estimate 95% confidence intervals around each monthly proportion.

**How bootstrapping works here:**
- For each month, resample the data 1000 times with replacement
- Compute the mean fire-risk proportion each time
- The 2.5th and 97.5th percentiles of these 1000 means form the 95% CI

Narrow CIs = pattern is reliable
Wide CIs = more uncertainty in that month's estimate

In [ ]:
n_boot = 1000                              # number of bootstrap samples
months = range(1, 13)
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
boot_means = {m: [] for m in months}      # store bootstrap results

for m in months:
    # Get all fire-risk values for this month
    month_data = df_clean[df_clean['Month'] == m]['ExtremeFire'].values

    # Resample 1000 times with replacement and compute mean each time
    B = np.random.choice(month_data,
                          size=(len(month_data), n_boot),
                          replace=True)    # bootstrap resampling
    boot_means[m] = np.mean(B, axis=0) * 100  # convert to percentage

# Calculate CI bounds and means
ci_low  = [np.percentile(boot_means[m], 2.5)  for m in months]   # lower bound
ci_high = [np.percentile(boot_means[m], 97.5) for m in months]   # upper bound
means   = [np.mean(boot_means[m])             for m in months]   # central estimate

# Plot with error bars showing 95% CI
plt.figure(figsize=(11, 4))
plt.bar(month_names, means, color='firebrick', alpha=0.7, label='Mean')
plt.errorbar(month_names, means,
             yerr=[np.array(means) - np.array(ci_low),
                   np.array(ci_high) - np.array(means)],
             fmt='none', color='black', capsize=4,
             label='95% Bootstrap CI')    # error bars show uncertainty
plt.xlabel('Month')
plt.ylabel('Extreme Fire-Risk Days (%)')
plt.title('Monthly Extreme Fire-Risk Days with 95% Bootstrap Confidence Intervals')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Discussion, Limitations, and Conclusion

### Summary of Findings

1. **Seasonal pattern:** Extreme fire-risk days peak in December–February
   (Australian summer), consistent with known bushfire seasonality.
   Bootstrapping confirmed this pattern is statistically robust.

2. **Bayesian two-group analysis:** High-temperature days receive
   significantly less rainfall. If the 95% HDI excludes zero, this
   provides strong evidence for the physical mechanism behind our
   fire-risk definition.

3. **Bambi regression + LOO-CV:** The combination of temperature,
   rainfall, and afternoon humidity was the strongest predictor set.
   Maximum temperature had the largest coefficient magnitude,
   making it the dominant driver of fire risk.

4. **Uncertainty quantification:** The Bayesian framework provided
   credible intervals around all estimates, rather than simple
   point estimates.

### Limitations

- **Fire-risk proxy:** `ExtremeFire` uses weather only. Fuel load,
  ignition sources, and terrain are absent from this analysis.
- **Subsampling:** 2,000 rows used for Bayesian models for
  computational feasibility.
- **Temporal gap:** Dataset ends in 2017. The 2019–2020 Black Summer
  — the most extreme season on record — is not included. Extending
  the analysis to include this season would be an important next step.
- **Normal likelihood:** Rainfall is right-skewed; a log-Normal or
  Gamma likelihood would be more appropriate for future work.

### Conclusion

Temperature, rainfall deficit, and afternoon humidity are strong
predictors of extreme fire-risk weather days in Australia. The
Bayesian framework allowed uncertainty to be quantified throughout.
Future work could incorporate actual fire event data, apply spatial
modelling by climate zone, and extend the dataset to include the
record-breaking 2019–2020 season.

## Personal Reflection

When choosing a topic for this project, I was drawn to Australian bushfires because they represent one of the most visible and devastating consequences of climate change. The 2019–2020 Black Summer bushfires left a strong impression on me due to their unprecedented scale and environmental impact. I wanted to investigate, from a data-driven perspective, which meteorological conditions were most strongly associated with periods of extreme fire risk.

My original plan was relatively straightforward: obtain a dataset containing both weather observations and fire occurrence records, apply the Bayesian methods learned in class, and analyse the relationship between environmental conditions and fire activity. In practice, however, the project became far more complicated than expected.

The first major challenge was data accessibility. I initially assumed that historical Australian bushfire records would be freely available and easy to integrate into statistical analysis. Instead, many official datasets required registration, were distributed mainly as GIS files incompatible with my workflow, or lacked the structure necessary for daily modelling. After considerable searching, I ultimately relied on two sources: the weatherAUS.csv dataset and daily rainfall data for Sydney Observatory Hill downloaded directly from the Australian Bureau of Meteorology.

The second challenge was methodological. Because I could not obtain reliable fire occurrence labels, I had to construct a proxy variable representing extreme fire-risk conditions. I defined the ExtremeFire label as days where maximum temperature exceeded the 90th percentile while rainfall was zero. Initially, I considered this a weakness because the variable could not directly represent actual fire occurrence. However, I later realised that defining and defending the proxy forced me to think more critically about what "fire risk" actually means and which assumptions were embedded within the analysis.

Another issue emerged when I discovered that the 10th percentile of rainfall in the dataset was exactly 0 mm, meaning my original threshold definition produced no fire-risk observations. Identifying and correcting this problem highlighted the importance of checking intermediate outputs rather than assuming the analysis pipeline was functioning correctly.

One strength of the final analysis is its methodological coherence. Each section builds logically on the previous one, and the Bayesian framework provides explicit uncertainty quantification rather than false precision. Working with Bayesian models also changed how I interpreted statistical results, encouraging me to view them as probabilistic estimates rather than fixed conclusions.

The main shortcoming remains the proxy nature of the ExtremeFire variable, which cannot fully capture actual fire occurrence or severity, and the need to subsample to 2,000 rows for computational efficiency.

Overall, this project showed me that real-world data analysis is far more iterative and uncertain than classroom exercises suggest. The gap between a clear research question and workable data required continuous methodological adaptation and careful justification of assumptions throughout the research process.